# Error Discovery — Focused Pipeline
**Goal:** maximise discovery of *new* error types and dimensions from agent transcripts.

Key design choices:
1. **No structured evaluation** — there are no fixed dimension scores. The LLM freely identifies what went wrong.
2. **Relaxed gate** — new types are invited when a failure is *meaningfully different* (different cause, policy impact, or user harm), not only as a last resort.
3. **Unconditional open-observation slot** — every transcript yields a free-form observations list, capturing candidate behaviors that don't yet have a name.
4. **Discovery analytics** — accumulation curve, per-dimension discovery rate, convergence check, proposals table, Markdown taxonomy export.

## 1. Imports & LLM Client

In [ ]:
import json, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import Optional, Callable
from pathlib import Path
from openai import AzureOpenAI

try:
    from IPython.display import display
except ImportError:
    display = print  # fallback for non-Jupyter environments

# Load credentials from .env if present (requires python-dotenv)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT", ""),
    api_key=os.getenv("AZURE_OPENAI_API_KEY", ""),
)

def call_llm(system_prompt: str, user_prompt: str, model: str = "gpt-4o",
             retries: int = 3) -> dict:
    """Call LLM with automatic retry on JSON parse failure."""
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ],
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            print(f"  [attempt {attempt+1}/{retries}] {e}")
            if attempt < retries - 1:
                time.sleep(1)
    return {}


## 2. Data Structures

In [ ]:
@dataclass
class ErrorType:
    name: str                          # CAPS_SNAKE_CASE
    definition: str                    # ≤15 words
    dimension: Optional[str] = None
    is_new: bool = False
    origin_task_id: Optional[str] = None


@dataclass
class Dimension:
    name: str
    question: str
    fail_guidance: str = ""
    pass_guidance: str = ""
    is_new: bool = False
    origin_task_id: Optional[str] = None


@dataclass
class ErrorRegistry:
    error_types: dict[str, ErrorType] = field(default_factory=dict)
    dimensions:  dict[str, Dimension] = field(default_factory=dict)
    discovery_log: list[dict] = field(default_factory=list)  # {task_id, name}

    def add_error_type(self, et: ErrorType, task_id: str = None):
        self.error_types[et.name] = et
        if task_id:
            self.discovery_log.append({"task_id": task_id, "name": et.name})

    def add_dimension(self, dim: Dimension):
        self.dimensions[dim.name] = dim

    def error_type_names(self) -> list[str]:
        return list(self.error_types.keys())

    def summary(self) -> dict:
        return {
            "total_error_types":  len(self.error_types),
            "base_error_types":   sum(1 for e in self.error_types.values() if not e.is_new),
            "new_error_types":    sum(1 for e in self.error_types.values() if e.is_new),
            "total_dimensions":   len(self.dimensions),
            "new_dimensions":     sum(1 for d in self.dimensions.values() if d.is_new),
        }


## 3. Base Registry
6 dimensions are seeded: 4 **Integrity** dimensions + 2 **Interaction** dimensions (`user_intent_adherence`, `user_question_fulfillment`).  
Removed from seed (left entirely open for discovery): `identity_accuracy`, `pii_safety`, `tone_appropriateness`, `action_execution_consistency`.  
Pass a subset via `applicable_dims` to further restrict the seed for a given domain.

In [ ]:
INTEGRITY_DIMS   = ["policy_violation", "policy_faithfulness",
                    "data_faithfulness", "tool_efficiency"]
INTERACTION_DIMS = ["user_intent_adherence", "user_question_fulfillment"]
ALL_BASE_DIMS    = INTERACTION_DIMS + INTEGRITY_DIMS


def build_base_registry(applicable_dims: Optional[list[str]] = None) -> ErrorRegistry:
    """
    Build the base registry with 6 seeded dimensions.
    applicable_dims: subset of ALL_BASE_DIMS to include (None = all 6).

    Intentionally excluded (left open for discovery):
      - identity_accuracy
      - pii_safety
      - tone_appropriateness
      - action_execution_consistency
    """
    active = set(applicable_dims or ALL_BASE_DIMS)
    registry = ErrorRegistry()

    base_error_types = [
        # Interaction
        ("USER_CONSTRAINT_VIOLATED",     "Ignored explicit user preference or constraint",           "user_intent_adherence"),
        ("USER_INPUT_MISREAD",           "Misread user-stated facts such as dates or amounts",       "user_intent_adherence"),
        ("QUESTION_UNANSWERED",          "User's question ignored or deflected without explanation", "user_question_fulfillment"),
        # Integrity
        ("HARMFUL_DISALLOWED_EXECUTION", "Executed forbidden state-changing action",                 "policy_violation"),
        ("DISALLOWED_DECISION",          "Agreed to forbidden action but did not execute",           "policy_violation"),
        ("MISSING_REQUIRED_CHECK",       "Skipped required verification before acting",              "policy_violation"),
        ("POLICY_HALLUCINATION",         "Stated policy rule that does not exist or is misstated",   "policy_faithfulness"),
        ("DATA_HALLUCINATION",           "Stated facts that contradict or are absent from tool output", "data_faithfulness"),
        ("REDUNDANT_IDENTICAL_CALL",     "Same function with same arguments called multiple times",  "tool_efficiency"),
        ("POST_COMPLETION_CALL",         "Made tool query after task was already resolved",          "tool_efficiency"),
        ("POST_IMPOSSIBILITY_CALL",      "Continued querying after confirming task is impossible",   "tool_efficiency"),
        ("UNNECESSARY_CALL",             "Tool query clearly irrelevant to user's request",          "tool_efficiency"),
    ]
    for name, defn, dim in base_error_types:
        if dim in active:
            registry.add_error_type(ErrorType(name=name, definition=defn, dimension=dim, is_new=False))

    base_dimensions = [
        ("user_intent_adherence",
         "Did the agent honor preferences and correctly read user input?",
         "Agent ignored explicit user preference OR misread user-stated facts.",
         "Policy overrides preference (policy wins); ambiguous input; offered alternatives."),
        ("user_question_fulfillment",
         "Did the agent answer the user's direct questions?",
         "Question ignored or deflected without explanation.",
         "Answered in different words; declined unauthorized request with explanation."),
        ("policy_violation",
         "Did the agent's actions and decisions follow policy?",
         "Agent executed, decided, or skipped a required check violating policy.",
         "Explored options before refusing; offered alternatives; discussed hypotheticals."),
        ("policy_faithfulness",
         "Did the agent correctly state policy rules?",
         "Agent stated a policy rule that doesn't exist or misstated a real rule.",
         "Simplified but directionally correct; reasonable paraphrases."),
        ("data_faithfulness",
         "Did the agent accurately report data from tool outputs?",
         "Agent stated facts that contradict or don't exist in tool outputs.",
         "Rounding ($99.99→$100); correct summarization."),
        ("tool_efficiency",
         "Were the agent's tool calls reasonable given available context?",
         "Agent made tool calls that were redundant, untimely, or irrelevant.",
         "Querying all records to find the right one; first-time queries."),
    ]
    for name, q, fg, pg in base_dimensions:
        if name in active:
            registry.add_dimension(Dimension(name=name, question=q, fail_guidance=fg, pass_guidance=pg, is_new=False))

    return registry


## 4. Discovery Prompt
For every transcript: identify all failures, flag which are already in the taxonomy, and propose new types when the failure is *meaningfully different*.  
Relaxed gate: "different cause, policy impact, or user harm" — not "last resort only".

In [ ]:
SYSTEM_PROMPT_DISCOVERY = """You are an expert agent failure analyst.
Your job is to identify every failure in the transcript and expand the error taxonomy
when you find something genuinely new.
Output ONLY valid JSON — no markdown, no text outside JSON."""


def build_discovery_prompt(
    policy: str,
    groundtruth: dict,
    transcript: dict,
    task_id: str,
    trial_id: str,
    registry: ErrorRegistry,
    ignored_behaviors: Optional[list[str]] = None,
) -> str:
    """
    Discovery prompt: run on every transcript.
    - Identifies all failures and maps them to existing error types where possible.
    - Proposes new types when the failure is *meaningfully different* from every existing one.
    - Always emits open_observations for anything worth noting.

    Relaxed gate: "meaningfully different" = different root cause, different policy impact,
    or a different kind of harm to the user — NOT a synonym or slight variation.
    """
    existing_block = "\n".join(
        f"  - {name} (→ {et.dimension or '?'}): {et.definition}"
        for name, et in registry.error_types.items()
    )
    existing_dims = "\n".join(
        f"  - {name}: {dim.question}"
        for name, dim in registry.dimensions.items()
    )

    ignored_block = ""
    if ignored_behaviors:
        items = "\n".join(f"  - {b}" for b in ignored_behaviors)
        ignored_block = f"\nBEHAVIORS TO IGNORE — do NOT flag these as failures:\n{items}\n"

    return f"""Analyse this agent transcript for failures.
{ignored_block}
{"="*76}
EXISTING ERROR TAXONOMY
{"="*76}

Error types:
{existing_block}

Dimensions:
{existing_dims}

{"="*76}
INPUTS
{"="*76}

POLICY:
{policy}

GROUND TRUTH:
{json.dumps(groundtruth, indent=2)}

TRANSCRIPT:
{json.dumps(transcript, indent=2)}

{"="*76}
INSTRUCTIONS
{"="*76}

Step 1 — List every failure you observe in "flagged_errors".
  For each failure, set "is_new": false if it matches an existing error type,
  or "is_new": true if it is meaningfully different.

Step 2 — For every failure where "is_new": true, add a full entry to "new_errors":
  {{
    "name":               "CAPS_SNAKE_CASE (max 3 words)",
    "definition":         "≤15 words — same terse style as existing types",
    "dimension":          "existing_dim_name OR new_snake_case_name",
    "dimension_question": "Did the agent X?" (only if brand-new dimension, else null),
    "evidence":           "Turn N: ≤20 words describing exactly what happened"
  }}
  A failure is "meaningfully different" when its root cause, policy implication,
  or user impact differs from every existing type — NOT just a synonym or variant.

Step 3 — Fill "open_observations" with anything else worth noting (unusual patterns,
  borderline behaviors, potential future taxonomy candidates).
  Leave as [] only if there is truly nothing extra.

{"="*76}
OUTPUT — return ONLY this JSON
{"="*76}

{{
  "task_id": "{task_id}",
  "trial_id": "{trial_id}",
  "flagged_errors": [
    {{"error_type": "<EXISTING_NAME or NEW_NAME>", "evidence": "Turn N: ...", "is_new": false}}
  ],
  "new_errors": [],
  "open_observations": []
}}
"""

## 5. Registry Updater
Processes `new_errors` from the LLM result and expands the registry in-place.
(The updater functions appear below the runner that calls them.)

## 6. Discovery Runner

In [ ]:
def run_discovery(
    policy: str,
    tasks: list[dict],
    simulations: list[dict],
    model: str = os.getenv("AZURE_OPENAI_MODEL", "gpt-4o"),
    applicable_dims: Optional[list[str]] = None,
    ignored_behaviors: Optional[list[str]] = None,
) -> dict:
    """
    Single-pass discovery loop — no structured dimension scoring.

    For each transcript the LLM:
      1. Flags all failures and maps them to existing error types where possible.
      2. Proposes new types when the failure is meaningfully different.
      3. Emits open_observations unconditionally.

    The registry grows in-place, so later tasks benefit from all types
    discovered on earlier ones.

    Returns:
        registry, all_results, registry_summary, accumulation_log
    """
    registry = build_base_registry(applicable_dims)
    all_results: list[dict] = []

    n = len(tasks)
    print(f"\n{'='*60}")
    print(f"Discovery run: {n} task(s) | model={model}")
    print(f"{'='*60}\n")

    for i, (task, sim) in enumerate(zip(tasks, simulations)):
        task_id  = str(task.get("id", i))
        trial_id = str(sim.get("trial", 0))
        reward   = sim.get("reward", sim.get("reward_info", {}).get("reward", None))
        if isinstance(reward, dict):
            reward = reward.get("reward", None)

        print(f"[{i+1}/{n}] Task {task_id} trial {trial_id}  reward={reward}")

        prompt = build_discovery_prompt(
            policy=policy,
            groundtruth=task,
            transcript=sim,
            task_id=task_id,
            trial_id=trial_id,
            registry=registry,
            ignored_behaviors=ignored_behaviors,
        )

        result = call_llm(SYSTEM_PROMPT_DISCOVERY, prompt, model)
        result["reward"] = reward
        all_results.append(result)

        grew = update_registry_from_discovery(registry, result, task_id)
        n_flagged = len(result.get("flagged_errors", []))
        n_new     = len(result.get("new_errors", []))
        n_obs     = len(result.get("open_observations", []))
        print(f"  flagged={n_flagged}  new_proposed={n_new}  observations={n_obs}  registry_grew={grew}")

    print(f"\nDone.  Registry: {registry.summary()}")
    return {
        "registry":         registry,
        "registry_summary": registry.summary(),
        "all_results":      all_results,
        "accumulation_log": registry.discovery_log,
    }


In [ ]:
def _register_new_error(registry: ErrorRegistry, entry: dict, task_id: str) -> bool:
    """Register a single new error type (+ dimension if needed). Returns True if registry grew."""
    grew  = False
    name  = (entry.get("name") or "").strip()
    defn  = entry.get("definition") or ""
    dim   = entry.get("dimension") or ""
    dim_q = entry.get("dimension_question")

    if not name or not dim:
        return False

    if dim not in registry.dimensions:
        q     = dim_q or f"Did the agent handle {dim.replace('_', ' ')} correctly?"
        label = dim.replace("_", " ")
        registry.add_dimension(Dimension(
            name=dim, question=q,
            fail_guidance=f"Agent failed to handle {label} correctly.",
            pass_guidance=f"Agent handled {label} appropriately.",
            is_new=True, origin_task_id=task_id,
        ))
        print(f"    [NEW DIMENSION] {dim}: {q}")
        grew = True

    if name not in registry.error_types:
        et = ErrorType(name=name, definition=defn, dimension=dim,
                       is_new=True, origin_task_id=task_id)
        registry.add_error_type(et, task_id=task_id)
        print(f"    [NEW ERROR] {name} (→ {dim}): {defn}")
        grew = True

    return grew


def update_registry_from_discovery(registry: ErrorRegistry, result: dict, task_id: str) -> bool:
    """Process new_errors list from the discovery result. Returns True if registry grew."""
    grew = False
    for entry in result.get("new_errors", []):
        if isinstance(entry, dict):
            grew |= _register_new_error(registry, entry, task_id)
    return grew


## 7. Utilities — Load Data & Save/Load Registry

In [ ]:
def load_tau2(json_path: str) -> tuple[list[dict], list[dict]]:
    """Load tasks and simulations from a tau2 benchmark results JSON."""
    with open(json_path) as f:
        data = json.load(f)
    tasks       = data.get("tasks", [])
    simulations = data.get("simulations", [])
    # align by task_id
    task_map = {str(t["id"]): t for t in tasks}
    paired_tasks, paired_sims = [], []
    for sim in simulations:
        tid = str(sim.get("task_id", ""))
        if tid in task_map:
            paired_tasks.append(task_map[tid])
            paired_sims.append(sim)
    print(f"Loaded {len(paired_tasks)} task-simulation pairs from {json_path}")
    return paired_tasks, paired_sims


def save_registry(registry: ErrorRegistry, path: str):
    data = {
        "error_types": {
            name: {"definition": et.definition, "dimension": et.dimension,
                   "is_new": et.is_new, "origin": et.origin_task_id}
            for name, et in registry.error_types.items()
        },
        "dimensions": {
            name: {"question": dim.question, "fail_guidance": dim.fail_guidance,
                   "pass_guidance": dim.pass_guidance, "is_new": dim.is_new, "origin": dim.origin_task_id}
            for name, dim in registry.dimensions.items()
        },
        "discovery_log": registry.discovery_log,
    }
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Registry saved → {path}  ({len(registry.error_types)} error types)")


def save_run_output(output: dict, path: str):
    serialisable = {k: v for k, v in output.items() if k != "registry"}
    with open(path, "w") as f:
        json.dump(serialisable, f, indent=2)
    print(f"Run output saved → {path}")


## 8. Configure & Run
Edit the config block below to point to your data files and choose your model.


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# Path to your tau2-bench results JSON (see sample_data/input_schema.json for the format)
JSON_PATH   = "./your_domain/results.json"

# Path to the domain policy Markdown file
POLICY_PATH = "./your_domain/policy.md"

# Output directory and file names
OUTPUT_DIR   = "./your_domain"
REGISTRY_OUT = f"{OUTPUT_DIR}/registry_focused.json"
RUN_OUT      = f"{OUTPUT_DIR}/run_focused.json"
TAXONOMY_OUT = f"{OUTPUT_DIR}/taxonomy_focused.md"

# LLM model name (must match a deployment in your Azure OpenAI resource)
MODEL = os.getenv("AZURE_OPENAI_MODEL", "gpt-4o")

# Set to a subset of ALL_BASE_DIMS to restrict seeded dimensions (None = all 6)
APPLICABLE_DIMS = None

# Behaviors the judge must never flag (add domain-specific quirks here)
IGNORED = [
    # Known benchmark artifact: the simulator sometimes generates concurrent calls
    # that are valid parallelism, not errors — suppress to reduce false positives.
    "concurrent tool calls (agent making more than one tool call in a single message)",
]
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(JSON_PATH):
    raise FileNotFoundError(
        f"Please update JSON_PATH to point to your data file.\n"
        f"  Current value: {JSON_PATH}\n"
        f"  See sample_data/input_schema.json for the expected format."
    )
tasks, sims = load_tau2(JSON_PATH)


In [ ]:
len(tasks), len(sims)

In [ ]:
output = run_discovery(
    policy=open(POLICY_PATH, encoding="utf-8").read(),
    tasks=tasks,
    simulations=sims,
    model=MODEL,
    applicable_dims=APPLICABLE_DIMS,
    ignored_behaviors=IGNORED,
)
save_registry(output["registry"], REGISTRY_OUT)
save_run_output(output, RUN_OUT)


## 9. Discovery Analytics

### 9a. Registry Summary & New Types Catalogue

In [ ]:
registry = output["registry"]
print(f"\nRegistry summary:")
for k, v in output["registry_summary"].items():
    print(f"  {k}: {v}")

print("\n── NEW ERROR TYPES ──────────────────────────────────────────────")
new_types = [(n, et) for n, et in registry.error_types.items() if et.is_new]
new_types.sort(key=lambda x: x[1].origin_task_id)
for name, et in new_types:
    print(f"  [{et.origin_task_id}] {name} (→ {et.dimension})")
    print(f"    {et.definition}")

print("\n── NEW DIMENSIONS ───────────────────────────────────────────────")
for name, dim in registry.dimensions.items():
    if dim.is_new:
        print(f"  [{dim.origin_task_id}] {name}: {dim.question}")


### 9b. Accumulation Curve — New Error Types Over Tasks
Shows *when* new types are found and whether discovery is converging.

In [ ]:
log = output["accumulation_log"]   # [{task_id, name}, ...]

if log:
    task_order = [r.get("task_id") for r in output["all_results"]]
    task_pos   = {tid: i for i, tid in enumerate(task_order)}

    cum = sorted(
        [(task_pos.get(str(e["task_id"]), -1), i + 1) for i, e in enumerate(log)],
        key=lambda x: x[0],
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    if cum:
        xs, ys = zip(*cum)
        ax.step(xs, ys, where="post", color="steelblue", linewidth=2, label="New error types")

    ax.set_xlabel("Task index")
    ax.set_ylabel("Cumulative new error types")
    ax.set_title("Error Type Discovery Accumulation Curve")
    ax.legend()
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Total new error types discovered: {len(log)}")
else:
    print("No new error types discovered — accumulation curve empty.")


### 9c. Discovery Rate Per Dimension
Which dimensions yielded the most new error types?

In [ ]:
dim_counts: Counter = Counter()
for name, et in registry.error_types.items():
    if et.is_new and et.dimension:
        dim_counts[et.dimension] += 1

if dim_counts:
    dims_sorted = sorted(dim_counts.keys(), key=lambda d: dim_counts[d], reverse=True)
    fig, ax = plt.subplots(figsize=(10, max(3, len(dims_sorted) * 0.5)))
    bars = ax.barh(dims_sorted[::-1], [dim_counts[d] for d in dims_sorted[::-1]], color="steelblue")
    ax.bar_label(bars, padding=3)
    ax.set_xlabel("New error types discovered")
    ax.set_title("New Error Types per Dimension")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()
else:
    print("No new error types to plot.")


### 9d. Open Observations Inventory
All `open_observations` collected during the run — useful source for manual taxonomy refinement.

In [ ]:
obs_rows = []
for r in output["all_results"]:
    task_id = r.get("task_id")
    reward  = r.get("reward")
    for obs in r.get("open_observations", []):
        obs_rows.append({
            "task_id":     task_id,
            "reward":      reward,
            "observation": obs,
        })

df_obs = pd.DataFrame(obs_rows)
print(f"Total open observations: {len(df_obs)}")
if not df_obs.empty:
    print(f"Tasks with observations: {df_obs['task_id'].nunique()}")
    display(df_obs)


### 9e. Proposals Table
All newly proposed error types across the run (accepted = added to registry; rejected = duplicate/synonym).

In [ ]:
proposal_rows = []
for r in output["all_results"]:
    if r is None:
        continue
    task_id = r.get("task_id")
    for p in r.get("new_errors", []):
        proposal_rows.append({
            "task_id":    task_id,
            "name":       p.get("name", ""),
            "definition": p.get("definition", ""),
            "dimension":  p.get("dimension", ""),
            "evidence":   p.get("evidence", ""),
            "accepted":   p.get("name", "") in registry.error_types,
        })

df_proposals = pd.DataFrame(proposal_rows)
print(f"Total proposals: {len(df_proposals)}")
if not df_proposals.empty:
    print(f"  Accepted (new to registry): {df_proposals['accepted'].sum()}")
    print(f"  Rejected (duplicate/synonym): {(~df_proposals['accepted']).sum()}")
    display(df_proposals.sort_values("accepted", ascending=False))


### 9f. Discovery Convergence Check
Are we still finding new types near the end of the run?  
A flat tail suggests convergence; a rising tail suggests more data is needed.

In [ ]:
log = output["accumulation_log"]
if len(log) >= 2:
    task_order   = [r.get("task_id") for r in output["all_results"]]
    task_pos_map = {tid: i for i, tid in enumerate(task_order)}

    positions = sorted(task_pos_map.get(str(e["task_id"]), 0) for e in log)
    n_tasks   = len(task_order)

    win     = max(1, n_tasks // 5)
    buckets = list(range(0, n_tasks, win))
    rates   = [sum(1 for p in positions if b <= p < b + win) for b in buckets]

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(range(len(rates)), rates, color="steelblue", alpha=0.7)
    ax.set_xticks(range(len(rates)))
    ax.set_xticklabels([f"{b}–{b+win}" for b in buckets], rotation=30, ha="right")
    ax.set_ylabel("New error types found")
    ax.set_title(f"Discovery rate per {win}-task window (convergence check)")
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

    last = rates[-1] if rates else 0
    print(f"Last window: {last} new type(s)")
    print("✓ Convergence likely." if last == 0 else "⚠ Discovery still active — consider more tasks.")
else:
    print("Not enough discovery events to assess convergence.")


## 10. Export Final Registry as Markdown Taxonomy

In [ ]:
def registry_to_markdown(registry: ErrorRegistry) -> str:
    """Render the full registry as a Markdown table, grouped by dimension."""
    by_dim: dict[str, list] = defaultdict(list)
    for name, et in registry.error_types.items():
        by_dim[et.dimension or "uncategorised"].append((name, et))

    lines = ["# Error Type Taxonomy\n"]
    dim_order = (
        [d for d in ALL_BASE_DIMS if d in by_dim]
        + [d for d in by_dim if d not in ALL_BASE_DIMS]
    )
    for dim in dim_order:
        ets = by_dim[dim]
        dim_info = registry.dimensions.get(dim)
        q = dim_info.question if dim_info else ""
        tag = " *(domain-specific)*" if (dim_info and dim_info.is_new) else ""
        lines.append(f"\n## {dim}{tag}\n*{q}*\n")
        lines.append("| Error Type | Definition | New | Origin |")
        lines.append("|---|---|:---:|---|")
        for name, et in sorted(ets, key=lambda x: x[1].is_new):
            new_tag = "✓" if et.is_new else ""
            lines.append(f"| `{name}` | {et.definition} | {new_tag} | {et.origin_task_id or ''} |")
    return "\n".join(lines)


md = registry_to_markdown(registry)
out_path = TAXONOMY_OUT
Path(out_path).write_text(md)
print(f"Taxonomy written to {out_path}\n")
print(md[:2000], "..." if len(md) > 2000 else "")
